# 02 - Build the embedding dataset 🧬

This notebook downloads both ONNX encoders, extracts embeddings, creates fixed train/validation/test splits, and uploads a Hugging Face dataset.

Rows are stored in CSV files. High-dimensional feature matrices are stored in compressed NPZ files beside the CSV files.

In [ ]:
%cd /content
![ -d Protein-Ligand-Affinity-Prediction-Using-LLM ] || git clone https://github.com/karthikeyanr103/Protein-Ligand-Affinity-Prediction-Using-LLM.git
%cd /content/Protein-Ligand-Affinity-Prediction-Using-LLM
!git pull
%pip install -e ".[runtime]" "huggingface-hub>=0.30"

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import HfApi, login, snapshot_download
from pathlib import Path
import os, subprocess

drive.mount('/content/drive')
HF_USER = 'your-huggingface-username'
PRO_REPO = f'{HF_USER}/prollama-affinity-onnx'
MOL_REPO = f'{HF_USER}/mol-llama-affinity-onnx'
DATASET_REPO = f'{HF_USER}/protein-compound-affinity-embeddings'
ROOT = Path('/content/drive/MyDrive/protein_affinity')
RAW_DATA = ROOT / 'data/train.csv'
CACHE = ROOT / 'embedding_cache'
DATASET_OUT = ROOT / 'embedding_dataset'
CACHE.mkdir(parents=True, exist_ok=True)
DATASET_OUT.mkdir(parents=True, exist_ok=True)
assert RAW_DATA.exists(), f'Missing: {RAW_DATA}'
token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = token
login(token=token)
PRO_ONNX = Path(snapshot_download(PRO_REPO, token=token))
MOL_ONNX = Path(snapshot_download(MOL_REPO, token=token))

## Inspect the source data

In [ ]:
from affinity.data import load_dataset, profile_dataset
frame = load_dataset(RAW_DATA)
profile_dataset(frame)

## Extract ONNX embeddings

Protein embeddings are stored in one file. Molecule embeddings are sharded so extraction can resume after a disconnect.

In [ ]:
subprocess.run([
    'affinity-extract-onnx',
    '--data', str(RAW_DATA),
    '--prollama-onnx', str(PRO_ONNX),
    '--mol-llama-onnx', str(MOL_ONNX),
    '--protein-output', str(CACHE / 'prollama_embeddings.npz'),
    '--molecule-output', str(CACHE / 'mol_llama'),
    '--protein-max-length', '1536',
    '--protein-batch-size', '1',
    '--molecule-shard-size', '1000',
], check=True)

## Create fixed train, validation and test files

In [ ]:
subprocess.run([
    'affinity-prepare-dataset',
    '--data', str(RAW_DATA),
    '--protein-embeddings', str(CACHE / 'prollama_embeddings.npz'),
    '--molecule-embeddings', str(CACHE / 'mol_llama'),
    '--output', str(DATASET_OUT),
    '--split-strategy', 'cold_protein',
    '--seed', '42',
], check=True)
print(sorted(path.name for path in DATASET_OUT.iterdir()))

In [ ]:
import pandas as pd, numpy as np, json
for split in ['train', 'validation', 'test']:
    rows = pd.read_csv(DATASET_OUT / f'{split}.csv')
    features = np.load(DATASET_OUT / f'{split}_features.npz')['features']
    print(split, 'rows=', len(rows), 'features=', features.shape)
print(json.loads((DATASET_OUT / 'dataset_metadata.json').read_text()))

## Upload the embedding dataset

In [ ]:
UPLOAD = False
if UPLOAD:
    api = HfApi(token=token)
    api.create_repo(DATASET_REPO, repo_type='dataset', exist_ok=True)
    api.upload_large_folder(
        repo_id=DATASET_REPO,
        repo_type='dataset',
        folder_path=str(DATASET_OUT),
    )
    print('Uploaded:', DATASET_REPO)